In [ ]:
# ============================================================
# FIX SUBMISSION HEADER: ID,caption -> ID,Caption
# ============================================================

from pathlib import Path
import pandas as pd
import zipfile

# Nếu bạn đang ở notebook gộp cuối cùng
SUBMISSION_CSV = Path("/kaggle/input/<FINAL_SUBMISSION_NOTEBOOK_OUTPUT>/qwen3_final_submission/submission.csv")

# Nếu không thấy file, tự tìm trong /kaggle/working
if not SUBMISSION_CSV.exists():
    candidates = list(Path("/kaggle/working").rglob("submission.csv"))
    print("Found candidates:")
    for c in candidates:
        print(c)
    if not candidates:
        raise FileNotFoundError("Không tìm thấy submission.csv trong /kaggle/working")
    SUBMISSION_CSV = candidates[0]

print("Using:", SUBMISSION_CSV)

df = pd.read_csv(SUBMISSION_CSV)

print("Before columns:", df.columns.tolist())

# Đổi đúng header theo yêu cầu evaluator
if "caption" in df.columns:
    df = df.rename(columns={"caption": "Caption"})

# Kiểm tra bắt buộc
if list(df.columns) != ["ID", "Caption"]:
    raise ValueError(f"Header vẫn sai. Current columns: {df.columns.tolist()}")

# Check dữ liệu
print("Rows:", len(df))
print("Duplicate IDs:", df["ID"].astype(str).duplicated().sum())
print("Empty captions:", df["Caption"].fillna("").astype(str).str.strip().eq("").sum())

# Save fixed CSV
OUT_DIR = Path("/kaggle/working/qwen3_final_submission_fixed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

fixed_csv = OUT_DIR / "submission.csv"
fixed_zip = OUT_DIR / "submission_qwen3_full_cui_cleaner_v4_FIXED_HEADER.zip"

df.to_csv(fixed_csv, index=False)

# Zip đúng format: bên trong zip phải là submission.csv
with zipfile.ZipFile(fixed_zip, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(fixed_csv, arcname="submission.csv")

print("Saved fixed CSV:", fixed_csv)
print("Saved fixed ZIP:", fixed_zip)

# Kiểm tra lại file trong zip
with zipfile.ZipFile(fixed_zip, "r") as z:
    print("Zip content:", z.namelist())
    with z.open("submission.csv") as f:
        head = pd.read_csv(f, nrows=3)
        print("Header in zip:", head.columns.tolist())
        display(head)